# Book Recommendation Models

This notebook builds several recommenders on the same book dataset and exposes them through a shared interface.

## Models included

1. **Hybrid item-to-item recommender**
   Combines collaborative similarity from user interactions with content similarity from book metadata.

2. **User-based KNN recommender**
   Uses explicit ratings to identify readers with similar taste.

3. **Biased matrix factorization recommender**
   Learns latent user and item factors together with user and item bias terms.

## Shared interface

Each recommender supports the same two main methods:

- `recommend_by_ids(seed_book_ids, ...)`
- `recommend_by_title(seed_titles, ...)`


In [1]:
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

import re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import normalize

ArrayLikeId = Union[str, int, float, Sequence[Union[str, int, float]]]
ArrayLikeTitle = Union[str, Sequence[str]]

## Load the cleaned datasets

The notebook expects two prepared tables:

- `books`, with one row per book and its metadata
- `ratings`, with user-book interactions in the form `(user_id, book_id, book_rating)`

In [2]:
books = pd.read_csv("../data/cleaned/cleaned_books.csv")
ratings = pd.read_csv("../data/cleaned/cleaned_ratings.csv")

print(f"Books shape:   {books.shape}")
print(f"Ratings shape: {ratings.shape}")

display(books.head(3))
display(ratings.head(3))

Books shape:   (271307, 11)
Ratings shape: (1031006, 4)


,isbn,title,author,year,publisher,image_url,book_id,implicit_ratings,explicit_ratings,avg_explicit_rating,total_ratings
0,1565920317,!%@ (A Nutshell handbook),Donnalyn Frey,1993.0,O'Reilly,http://images.amazon.com/images/P/1565920317.0...,1,0,1,6.0,1
1,1565920465,!%@ (A Nutshell handbook),Donnalyn Frey,1994.0,O'Reilly,http://images.amazon.com/images/P/1565920465.0...,1,1,0,NaN,1
2,0133989429,!Arriba! Comunicacion y cultura,Eduardo Zayas-Bazan,1996.0,Prentice Hall,http://images.amazon.com/images/P/0133989429.0...,2,1,1,9.0,2


,user_id,isbn,book_rating,book_id
0,276725,034545104X,0,69935
1,276726,0155061224,5,155136
2,276727,0446520802,0,207251


## Shared utilities

This section contains the common infrastructure used by all models:

- **identifier and title normalization**
- **text cleaning**
- **catalog construction**

  All recommenders need a shared metadata table describing books.
  `build_book_catalog(...)` creates this reference catalog, ensures required columns exist, normalizes identifiers, builds a normalized title key, and keeps one best metadata row per book
- **interaction preparation**

  Collaborative filtering models need a clean explicit-feedback table.
  `prepare_explicit_interactions(...)` filters the ratings data, removes invalid rows, merges repeated user-item interactions, and drops users or books with too little signal.
- **common recommender workflow**

  `SeedBookRecommenderBase` defines the reusable logic shared by all seed-based recommenders:
  - storing the catalog
  - resolving titles to IDs
  - validating that the model is fitted
  - normalizing user input
  - excluding seed books from the output
  - formatting the final recommendation table


In [3]:
def normalize_book_id_value(value: Any) -> str:
    """
    Convert book identifiers to a stable string form.
    """
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text == "":
        return ""
    try:
        numeric = float(text)
        if numeric.is_integer():
            return str(int(numeric))
    except Exception:
        pass
    return text


def normalize_book_id_series(s: pd.Series) -> pd.Series:
    """
    Normalize all book identifiers in a pandas Series.
    """
    return s.map(normalize_book_id_value)


def normalize_title(text: Any) -> str:
    """
    Convert a title into a normalized lookup form.
    """
    return re.sub(r"\s+", " ", str(text).strip().lower())


def clean_text_series(s: pd.Series) -> pd.Series:
    """
    Lowercase and whitespace-normalize a text Series.
    """
    return (
        s.fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def build_book_catalog(
    books: pd.DataFrame,
    *,
    id_col: str = "book_id",
    title_col: str = "title",
    author_col: str = "author",
    publisher_col: str = "publisher",
    isbn_col: str = "isbn",
    year_col: str = "year",
    image_url_col: str = "image_url",
    popularity_col: str = "explicit_ratings",
    rating_meta_col: str = "avg_explicit_rating",
) -> pd.DataFrame:
    """
    Create one clean metadata row per book.

    The catalog produced here is the shared reference table used by all models.
    It keeps one row per ``book_id``, ensures required metadata columns exist,
    creates a normalized title key for title-based lookup, and sorts duplicate
    records so that the best-supported row is kept.
    """
    required = [id_col, title_col]
    missing = [c for c in required if c not in books.columns]
    if missing:
        raise ValueError(f"`books` is missing required columns: {missing}")

    df = books.copy()
    df = df[df[id_col].notna()].copy()
    df[id_col] = normalize_book_id_series(df[id_col])

    for col in [title_col, author_col, publisher_col, isbn_col, image_url_col]:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].fillna("").astype(str)

    if year_col not in df.columns:
        df[year_col] = np.nan
    df[year_col] = df[year_col].astype("Int32")

    if popularity_col not in df.columns:
        df[popularity_col] = 0
    df[popularity_col] = df[popularity_col].astype("Int32").fillna(0)

    if rating_meta_col not in df.columns:
        df[rating_meta_col] = np.nan
    df[rating_meta_col] = pd.to_numeric(df[rating_meta_col], errors="coerce")

    df["__title_key"] = df[title_col].map(normalize_title)

    df = (
        df.sort_values(
            by=[popularity_col, rating_meta_col, title_col],
            ascending=[False, False, True],
            na_position="last",
        )
        .drop_duplicates(subset=[id_col], keep="first")
        .reset_index(drop=True)
    )
    return df


def prepare_explicit_interactions(
    ratings: pd.DataFrame,
    *,
    user_col: str = "user_id",
    item_col: str = "book_id",
    rating_col: str = "book_rating",
    explicit_min: float = 1.0,
    explicit_max: float = 10.0,
    min_user_ratings: int = 10,
    min_item_ratings: int = 10,
) -> pd.DataFrame:
    """
    Prepare the explicit-feedback interaction table used by explicit models.

    Processing steps:
    1. keep only the required columns
    2. drop rows with missing user or item IDs
    3. convert ratings to numeric values
    4. keep explicit ratings above ``explicit_min``
    5. merge repeated user-item rows by averaging their ratings
    6. remove users and items that do not satisfy the minimum
       interaction thresholds
    """
    required = [user_col, item_col, rating_col]
    missing = [c for c in required if c not in ratings.columns]
    if missing:
        raise ValueError(f"`ratings` is missing required columns: {missing}")

    df = ratings[[user_col, item_col, rating_col]].copy()
    df = df[df[user_col].notna() & df[item_col].notna()].copy()

    df[user_col] = df[user_col].astype(str)
    df[item_col] = normalize_book_id_series(df[item_col])
    df[rating_col] = df[rating_col].astype("Int32")

    df = df[df[rating_col] >= explicit_min].copy()
    df = df[df[rating_col] <= explicit_max].copy()
    df = df.groupby([user_col, item_col], as_index=False)[rating_col].mean()


    keep_users = df[user_col].value_counts()
    keep_users = keep_users[keep_users >= min_user_ratings].index

    keep_items = df[item_col].value_counts()
    keep_items = keep_items[keep_items >= min_item_ratings].index

    df = df[df[user_col].isin(keep_users)].copy()
    df = df[df[item_col].isin(keep_items)].copy()

    return df.reset_index(drop=True)


class SeedBookRecommenderBase:
    """
    Base class for recommenders that generate suggestions from one or more seed books.

    This class implements the shared workflow used by all seed-based recommenders.
    It is responsible for:

    - building and storing the shared book catalog
    - validating that the model has been fitted
    - normalizing seed book IDs
    - resolving seed titles to IDs
    - mapping catalog IDs to row positions
    - formatting final recommendation outputs
    """

    def __init__(
        self,
        *,
        id_col: str = "book_id",
        title_col: str = "title",
        author_col: str = "author",
        publisher_col: str = "publisher",
        isbn_col: str = "isbn",
        year_col: str = "year",
        image_url_col: str = "image_url",
        popularity_col: str = "explicit_ratings",
        rating_meta_col: str = "avg_explicit_rating",
    ):
        self.id_col = id_col
        self.title_col = title_col
        self.author_col = author_col
        self.publisher_col = publisher_col
        self.isbn_col = isbn_col
        self.year_col = year_col
        self.image_url_col = image_url_col
        self.popularity_col = popularity_col
        self.rating_meta_col = rating_meta_col

        self.catalog_: Optional[pd.DataFrame] = None
        self.catalog_id_to_pos_: Optional[Dict[str, int]] = None

    def _set_catalog(self, books: pd.DataFrame) -> None:
        """
        Build and store the shared catalog used for lookup and output formatting.
        """
        self.catalog_ = build_book_catalog(
            books,
            id_col=self.id_col,
            title_col=self.title_col,
            author_col=self.author_col,
            publisher_col=self.publisher_col,
            isbn_col=self.isbn_col,
            year_col=self.year_col,
            image_url_col=self.image_url_col,
            popularity_col=self.popularity_col,
            rating_meta_col=self.rating_meta_col,
        )
        self.catalog_id_to_pos_ = {
            str(book_id): idx for idx, book_id in enumerate(self.catalog_[self.id_col].tolist())
        }

    def _check_is_fitted(self) -> None:
        """
        Raise an error if the model has not been fitted yet.
        """
        if self.catalog_ is None:
            raise RuntimeError("The recommender is not fitted yet. Call fit(...) first.")

    def _normalize_seed_ids(self, seed_book_ids: ArrayLikeId) -> List[str]:
        """
        Normalize one or more user-provided seed book IDs.

        Parameters
        ----------
        seed_book_ids : ArrayLikeId
            A single seed ID or a sequence of seed IDs. Accepted forms include
            strings, integers, floats, or sequences of those values.

        Returns
        -------
        List[str]
            A list of normalized non-empty book ID strings.

        """
        if isinstance(seed_book_ids, (str, int, float, np.integer, np.floating)):
            seed_book_ids = [seed_book_ids]
        out = [normalize_book_id_value(x) for x in seed_book_ids]
        out = [x for x in out if x]
        if not out:
            raise ValueError("No valid seed book IDs were provided.")
        return out

    def _resolve_titles_to_ids(
        self,
        seed_titles: ArrayLikeTitle,
        *,
        top_k_per_title: int = 1,
    ) -> List[str]:
        """
        Resolve one or more seed titles to catalog book IDs.

        Titles are normalized before matching. If multiple catalog rows share
        the same normalized title, the method returns the best-supported match
        or matches according to catalog popularity and rating metadata.

        Parameters
        ----------
        seed_titles : ArrayLikeTitle
            A single title or a sequence of titles to resolve.

        top_k_per_title : int, default=1
            Maximum number of matching IDs to return for each provided title.

        Returns
        -------
        List[str]
            A list of resolved catalog book IDs.
        """
        self._check_is_fitted()
        titles = [seed_titles] if isinstance(seed_titles, str) else list(seed_titles)
        out: List[str] = []

        for title in titles:
            key = normalize_title(title)
            matches = self.catalog_[self.catalog_["__title_key"] == key]
            if matches.empty:
                continue

            matches = matches.sort_values(
                by=[self.popularity_col, self.rating_meta_col, self.title_col],
                ascending=[False, False, True],
                na_position="last",
            )
            out.extend(matches[self.id_col].astype(str).head(top_k_per_title).tolist())

        if not out:
            raise ValueError("None of the provided titles were found in the catalog.")
        return out

    def _catalog_positions(self, book_ids: Sequence[str]) -> List[int]:
        """
        Convert catalog book IDs into integer row positions.

        Parameters
        ----------
        book_ids : Sequence[str]
            Sequence of normalized book IDs.

        Returns
        -------
        List[int]
            Row positions of the IDs that exist in the current catalog.
        """
        return [
            self.catalog_id_to_pos_[book_id]
            for book_id in book_ids
            if book_id in self.catalog_id_to_pos_
        ]

    def _build_output(
        self,
        scores: np.ndarray,
        *,
        seed_ids: Sequence[str],
        n: int,
        exclude_input: bool = True,
        component_scores: Optional[Dict[str, np.ndarray]] = None,
    ) -> pd.DataFrame:
        """
        Convert a catalog-aligned score vector into a recommendation table.

        Parameters
        ----------
        scores : np.ndarray
            Score vector aligned with the rows of ``self.catalog_``.
            Each position corresponds to one catalog book.

        seed_ids : Sequence[str]
            Normalized seed book IDs used to generate the recommendations.

        n : int
            Number of recommendations to return.

        exclude_input : bool, default=True
            Whether to remove the seed books themselves from the output.

        component_scores : dict[str, np.ndarray] or None, default=None
            Optional dictionary of additional score vectors aligned with the
            catalog, for example separate collaborative and content components.

        Returns
        -------
        pd.DataFrame
            A formatted recommendation table containing metadata columns,
            the final score, and optionally the component scores.

        """
        self._check_is_fitted()

        scores = np.asarray(scores, dtype=np.float64).copy()
        if len(scores) != len(self.catalog_):
            raise ValueError("Scores must be aligned with the catalog.")

        if exclude_input:
            for pos in self._catalog_positions(seed_ids):
                scores[pos] = -np.inf

        order = np.argsort(-scores)
        valid = np.isfinite(scores[order])
        order = order[valid][:n]

        cols = [
            self.id_col,
            self.title_col,
            self.author_col,
            self.publisher_col,
            self.year_col,
            self.isbn_col,
            self.image_url_col,
            self.popularity_col,
            self.rating_meta_col,
        ]
        cols = [c for c in cols if c in self.catalog_.columns]

        out = self.catalog_.iloc[order][cols].copy()
        out["score"] = scores[order].round(3)

        if component_scores:
            for name, arr in component_scores.items():
                out[name] = np.asarray(arr)[order]

        return out.reset_index(drop=True)

    def recommend_by_ids(
        self,
        seed_book_ids: ArrayLikeId,
        *,
        n: int = 10,
        seed_ratings: Optional[Sequence[float]] = None,
        exclude_input: bool = True,
        return_components: bool = False,
        **kwargs: Any,
    ) -> pd.DataFrame:
        """
        Generate recommendations from one or more seed book IDs.

        Parameters
        ----------
        seed_book_ids : ArrayLikeId
            A single seed ID or a sequence of seed IDs.

        n : int, default=10
            Number of recommendations to return.

        seed_ratings : Sequence[float] or None, default=None
            Optional ratings associated with the seed books. Subclasses may use
            them to weight the influence of individual seeds.

        exclude_input : bool, default=True
            Whether to exclude the seed books from the final recommendation list.

        return_components : bool, default=False
            Whether to include model-specific component score columns in the
            output when available.

        **kwargs : Any
            Additional keyword arguments forwarded to the subclass scoring method.

        Returns
        -------
        pd.DataFrame
            A formatted recommendation table.
        """
        self._check_is_fitted()
        seed_ids = self._normalize_seed_ids(seed_book_ids)
        scores, components = self._score_from_seed_ids(
            seed_ids,
            seed_ratings=seed_ratings,
            return_components=return_components,
            **kwargs,
        )
        return self._build_output(
            scores,
            seed_ids=seed_ids,
            n=n,
            exclude_input=exclude_input,
            component_scores=components if return_components else None,
        )

    def recommend_by_title(
        self,
        seed_titles: ArrayLikeTitle,
        *,
        n: int = 10,
        seed_ratings: Optional[Sequence[float]] = None,
        exclude_input: bool = True,
        top_k_per_title: int = 1,
        return_components: bool = False,
        **kwargs: Any,
    ) -> pd.DataFrame:
        """
        Recommend books from one or more seed titles.
        """
        seed_ids = self._resolve_titles_to_ids(seed_titles, top_k_per_title=top_k_per_title)
        return self.recommend_by_ids(
            seed_ids,
            n=n,
            seed_ratings=seed_ratings,
            exclude_input=exclude_input,
            return_components=return_components,
            **kwargs,
        )

    def _score_from_seed_ids(
        self,
        seed_ids: Sequence[str],
        *,
        seed_ratings: Optional[Sequence[float]] = None,
        return_components: bool = False,
        **kwargs: Any,
    ) -> Tuple[np.ndarray, Optional[Dict[str, np.ndarray]]]:
        """
        Subclasses must return a score vector aligned with ``self.catalog_``.
        """
        raise NotImplementedError


## Hybrid item-to-item KNN recommender

`HybridItemKNNRecommender` combines two sources of similarity between books:

1. **collaborative similarity** from user ratings
2. **content similarity** from book metadata represented with TF-IDF

The final recommendation score is a weighted combination of both parts.

It also supports:

- **signed ratings**, so dislikes can reduce the score
- **optional normalization of component scores**
- **optional MMR reranking** for more diverse recommendations

---

### 1. Signed rating transformation

Ratings are centered around a neutral point `neutral_rating`.

Let:

- $r$ = rating
- $n$ = neutral rating

Then:

$$
s(r)=
\begin{cases}
\frac{r-n}{10-n}, & r \ge n \\\\
\frac{r-n}{n-1}, & r < n
\end{cases}
$$

This maps ratings approximately into $[-1, 1]$:

- positive ratings $\rightarrow$ positive values
- neutral ratings $\rightarrow$ near 0
- negative ratings $\rightarrow$ negative values

A deadzone removes weak signals near neutral:

$$
s_d(r)=
\begin{cases}
0, & |s(r)| < d \\\\
s(r), & \text{otherwise}
\end{cases}
$$

---

### 2. Collaborative item representation

Training ratings are converted into signed collaborative signals:

$$
g(r)=
\begin{cases}
w_{\text{impl}}, & r=0 \\\\
w_{\text{exp}} \cdot s_d(r), & r>0
\end{cases}
$$

where:

- $w_{\text{impl}}$ = `implicit_weight`
- $w_{\text{exp}}$ = `explicit_weight`

These values are used to build an item-user matrix:

$$
X \in \mathbb{R}^{I \times U}
$$

Each item vector is L2-normalized and item-item collaborative similarity is cosine similarity:

$$
\text{sim}_{\text{collab}}(i,j)=\hat{x}_i^\top \hat{x}_j
$$

Because the signals are signed, collaborative similarity can be positive or negative.

---

### 3. Content representation

For each book, a text document is built from metadata such as:

- title
- author
- publisher

The fields can be repeated with different weights:

$$
\text{document}_i =
(\text{title}_i)^{\times w_t}
+(\text{author}_i)^{\times w_a}
+(\text{publisher}_i)^{\times w_p}
$$

The corpus is vectorized with TF-IDF:

$$
T \in \mathbb{R}^{I \times F}
$$

and item-item content similarity is again cosine similarity:

$$
\text{sim}_{\text{content}}(i,j)=\hat{t}_i^\top \hat{t}_j
$$

---

### 4. Query with one or more seed books

At recommendation time, each seed can also have an optional rating.
That rating is converted into a signed seed weight:

$$
a_m = s_{d,\text{seed}}(r_m)
$$

If no seed ratings are given, all seeds get equal positive weights.

For a candidate item $j$, the collaborative and content scores are aggregated over all seeds:

$$
C_j =
\frac{\sum_{m=1}^{M} a_m \cdot \text{sim}_{\text{collab}}(i_m, j)}
{\sum_{m=1}^{M} |a_m|}
$$

$$
T_j =
\frac{\sum_{m=1}^{M} a_m \cdot \text{sim}_{\text{content}}(i_m, j)}
{\sum_{m=1}^{M} |a_m|}
$$

So:

- liked seeds push similar items up
- disliked seeds push similar items down

---

### 5. Final hybrid score

The two components are optionally normalized and then combined:

$$
H_j = \alpha C_j + \beta T_j
$$

where:

- $\alpha$ = `collaborative_weight`
- $\beta$ = `content_weight`

This gives the final base recommendation score.

---

### 6. Optional diversity reranking with MMR

To avoid returning many very similar books, the recommender can rerank the top candidates with **Maximal Marginal Relevance (MMR)**:

$$
\text{MMR}(j)=
\lambda \cdot \text{Rel}(j)
-
(1-\lambda)\cdot
\max_{s \in S}\text{sim}_{\text{content}}(j,s)
$$

where:

- $\text{Rel}(j)$ is the base hybrid relevance
- $S$ is the set of already selected recommendations
- $\lambda$ controls relevance versus diversity

Higher $\lambda$ favors relevance more, while lower $\lambda$ favors diversity more.

---

This method recommends books that are similar to the seed books both in **user preference patterns** and in **metadata content**, while also allowing dislikes and optional diversity control.

In [4]:
class HybridItemKNNRecommender(SeedBookRecommenderBase):
    """
    Hybrid item-to-item recommender that combines collaborative and content signals.

    The model builds two item representations:

    1. a collaborative item-user representation derived from signed rating signals
    2. a content representation derived from TF-IDF features of book metadata

    For a recommendation query, each seed book contributes both collaborative and
    content similarity scores. These scores are aggregated across seeds, optionally
    normalized per query, and then combined into a final hybrid score.

    The class also supports:
    - signed query-time seed weights from seed ratings
    - keeping only the strongest similarities per seed
    - optional Maximal Marginal Relevance reranking for diversity

    Parameters
    ----------
    user_col : str, default="user_id"
    item_col : str, default="book_id"
    rating_col : str, default="book_rating"

    implicit_weight : float, default=1.0
        Strength assigned to implicit interactions encoded as rating 0.

    explicit_weight : float, default=2.5
        Scaling factor applied to explicit ratings after centering them around

    neutral_rating : float, default=6.0
        Rating value treated as neutral. Ratings above it become positive and
        ratings below it become negative.

    signal_deadzone : float, default=0.1
        Threshold used when converting training ratings into signed collaborative
        signals. Values whose absolute centered magnitude is below this threshold
        are set to zero.

    seed_deadzone : float, default=0.1
        Threshold used when converting query-time seed ratings into signed seed
        weights. Values near neutral are suppressed.

    min_user_interactions : int, default=2
        Minimum number of retained interactions required for a user to remain in
        the collaborative training table.

    min_item_interactions : int, default=3
        Minimum number of retained interactions required for an item to remain in
        the collaborative training table.

    collaborative_weight : float, default=0.70
        Weight of the collaborative component in the final hybrid score.

    content_weight : float, default=0.30
        Weight of the content component in the final hybrid score.

    component_norm : {"maxabs", "zscore"} or None, default="maxabs"
        Optional per-query normalization applied separately to collaborative and
        content score vectors before they are mixed.

    max_features : int, default=100_000
        Maximum number of TF-IDF features retained by the vectorizer.

    min_df : int, default=2
        Minimum document frequency used by the TF-IDF vectorizer.

    ngram_range : tuple[int, int], default=(1, 2)
        Lower and upper n-gram range used by the TF-IDF vectorizer.

    diversity_lambda : float or None, default=0.80
        Relevance-diversity tradeoff used by MMR reranking.
        Higher values favor relevance more strongly.
        If None, diversity reranking is disabled.

    rerank_pool : int, default=150
        Number of top base candidates considered during MMR reranking.

    top_k_by_abs : bool, default=True
        If True, keep the strongest similarities by absolute value when applying
        the per-seed top-k mask.
    """

    def __init__(
        self,
        *,
        user_col: str = "user_id",
        item_col: str = "book_id",
        rating_col: str = "book_rating",
        implicit_weight: float = 1.0,
        explicit_weight: float = 2.5,
        neutral_rating: float = 6.0,
        signal_deadzone: float = 0.5,
        seed_deadzone: float = 0.5,
        min_user_interactions: int = 2,
        min_item_interactions: int = 3,
        collaborative_weight: float = 0.70,
        content_weight: float = 0.30,
        component_norm: Optional[str] = "maxabs",
        max_features: int = 100_000,
        min_df: int = 2,
        ngram_range: Tuple[int, int] = (1, 2),
        diversity_lambda: Optional[float] = 0.80,
        rerank_pool: int = 150,
        top_k_by_abs: bool = True,
        **kwargs: Any,
    ):
        super().__init__(**kwargs)
        self.user_col = user_col
        self.item_col = item_col
        self.rating_col = rating_col

        self.implicit_weight = implicit_weight
        self.explicit_weight = explicit_weight
        self.neutral_rating = neutral_rating
        self.signal_deadzone = signal_deadzone
        self.seed_deadzone = seed_deadzone

        self.min_user_interactions = min_user_interactions
        self.min_item_interactions = min_item_interactions

        self.collaborative_weight = collaborative_weight
        self.content_weight = content_weight
        self.component_norm = component_norm

        self.max_features = max_features
        self.min_df = min_df
        self.ngram_range = ngram_range

        self.diversity_lambda = diversity_lambda
        self.rerank_pool = rerank_pool
        self.top_k_by_abs = top_k_by_abs

        self.item_user_norm_: Optional[csr_matrix] = None
        self.content_norm_: Optional[csr_matrix] = None
        self.vectorizer_: Optional[TfidfVectorizer] = None

    def _centered_signed_value(
        self,
        rating: float,
        *,
        deadzone: float,
    ) -> float:
        """
        Convert a rating on the 1 to 10 scale into a signed value around the
        configured neutral rating.

        Ratings above `neutral_rating` become positive, ratings below it become
        negative, and the output is scaled to approximately the interval [-1, 1].
        A deadzone is then applied so values close to neutral become exactly zero.
        """
        rating = float(np.clip(rating, 1.0, 10.0))

        if rating >= self.neutral_rating:
            scale = max(10.0 - self.neutral_rating, 1e-9)
            value = (rating - self.neutral_rating) / scale
        else:
            scale = max(self.neutral_rating - 1.0, 1e-9)
            value = (rating - self.neutral_rating) / scale

        if abs(value) < deadzone:
            value = 0.0
        return float(value)

    def _rating_to_signal(self, rating: float) -> float:
        """
        Convert a raw training rating into a signed collaborative signal.

        Explicit ratings are centered around `neutral_rating` and scaled by
        `explicit_weight`. Implicit interactions encoded as rating 0 are mapped
        to a weak positive signal controlled by `implicit_weight`.
        """
        if rating == 0:
            return float(self.implicit_weight)

        signed = self._centered_signed_value(rating, deadzone=self.signal_deadzone)
        return float(self.explicit_weight * signed)

    def _build_content_corpus(
        self,
        catalog: pd.DataFrame,
        title_weight: int = 3,
        author_weight: int = 2,
        publisher_weight: int = 1,
    ) -> pd.Series:
        """
        Build the metadata text corpus used by the TF-IDF content model.

        The corpus is formed by concatenating selected metadata fields and
        repeating them according to their weights. Repetition makes a field more
        influential in the resulting TF-IDF representation.
        """
        title_text = catalog[self.title_col].fillna("").astype(str)
        author_text = catalog[self.author_col].fillna("").astype(str)
        publisher_text = catalog[self.publisher_col].fillna("").astype(str)

        corpus = (
            ("title " + title_text + " ") * title_weight
            + ("author " + author_text + " ") * author_weight
            + ("publisher " + publisher_text + " ") * publisher_weight
        )
        return corpus.str.replace(r"\s+", " ", regex=True).str.strip()

    def fit(self, books: pd.DataFrame, ratings: pd.DataFrame) -> "HybridItemKNNRecommender":
        """
        Fit the hybrid recommender on books and ratings.

        Collaborative part:
        - build a signed item-user matrix from ratings
        - row-normalize it so cosine similarity can be used

        Content part:
        - build TF-IDF vectors from metadata
        - row-normalize them

        Parameters
        ----------
        books : pd.DataFrame
            Book metadata table used to build the shared catalog and content model.

        ratings : pd.DataFrame
            User-item-rating interaction table used to build the collaborative
            item-user matrix.

        Returns
        -------
        HybridItemKNNRecommender
            The fitted recommender instance.
        """
        self._set_catalog(books)
        catalog = self.catalog_

        df = ratings[[self.user_col, self.item_col, self.rating_col]].copy()
        df = df[df[self.user_col].notna() & df[self.item_col].notna()].copy()

        df[self.user_col] = df[self.user_col].astype(str)
        df[self.item_col] = normalize_book_id_series(df[self.item_col])
        df[self.rating_col] = pd.to_numeric(df[self.rating_col], errors="coerce").fillna(0.0)

        valid_ids = set(catalog[self.id_col].astype(str))
        df = df[df[self.item_col].isin(valid_ids)].copy()

        df = (
            df.groupby([self.user_col, self.item_col], as_index=False)[self.rating_col]
            .mean()
        )

        df["signal"] = df[self.rating_col].map(self._rating_to_signal)

        df = df[np.abs(df["signal"]) > 0].copy()

        changed = True
        while changed and not df.empty:
            before = len(df)

            if self.min_user_interactions > 1:
                keep_users = df[self.user_col].value_counts()
                keep_users = keep_users[keep_users >= self.min_user_interactions].index
                df = df[df[self.user_col].isin(keep_users)].copy()

            if self.min_item_interactions > 1:
                keep_items = df[self.item_col].value_counts()
                keep_items = keep_items[keep_items >= self.min_item_interactions].index
                df = df[df[self.item_col].isin(keep_items)].copy()

            changed = len(df) != before

        if df.empty:
            raise ValueError("No interactions remain after cleaning the hybrid feedback table.")

        user_codes, user_uniques = pd.factorize(df[self.user_col], sort=True)
        item_pos = df[self.item_col].map(self.catalog_id_to_pos_).to_numpy()
        values = df["signal"].to_numpy(np.float32)

        item_user = csr_matrix(
            (values, (item_pos, user_codes)),
            shape=(len(catalog), len(user_uniques)),
            dtype=np.float32,
        )
        self.item_user_norm_ = normalize(item_user, norm="l2", axis=1)

        corpus = self._build_content_corpus(catalog)
        self.vectorizer_ = TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            stop_words="english",
            min_df=self.min_df,
            ngram_range=self.ngram_range,
            max_features=self.max_features,
            sublinear_tf=True,
        )
        content = self.vectorizer_.fit_transform(corpus)
        self.content_norm_ = normalize(content, norm="l2", axis=1)

        return self

    def _prepare_valid_seeds(
        self,
        seed_ids: Sequence[str],
        seed_ratings: Optional[Sequence[float]],
    ) -> Tuple[List[str], np.ndarray, Optional[np.ndarray]]:
        """
        Keep only seed books that exist in the catalog and preserve alignment with
        the optional seed ratings.

        Parameters
        ----------
        seed_ids : Sequence[str]
            Normalized seed book IDs provided by the user.

        seed_ratings : Sequence[float] or None
            Optional ratings associated with the seed books.

        Returns
        -------
        tuple[list[str], np.ndarray, np.ndarray or None]
            A tuple containing:
            1. valid seed IDs
            2. catalog row positions of those seeds
            3. aligned seed ratings if provided, otherwise None
        """
        if seed_ratings is not None and len(seed_ratings) != len(seed_ids):
            raise ValueError("seed_ratings must have the same length as seed_book_ids.")

        valid_ids: List[str] = []
        valid_positions: List[int] = []
        valid_ratings: List[float] = []

        for i, book_id in enumerate(seed_ids):
            pos = self.catalog_id_to_pos_.get(book_id)
            if pos is None:
                continue
            valid_ids.append(book_id)
            valid_positions.append(pos)
            if seed_ratings is not None:
                valid_ratings.append(float(seed_ratings[i]))

        if not valid_positions:
            raise KeyError("None of the provided seed book IDs exist in the catalog.")

        rating_arr = None
        if seed_ratings is not None:
            rating_arr = np.asarray(valid_ratings, dtype=np.float32)

        return valid_ids, np.asarray(valid_positions, dtype=np.int64), rating_arr

    def _seed_weights(self, seed_ratings: Optional[Sequence[float]], n_seeds: int) -> np.ndarray:
        """
        Convert optional query-time seed ratings into signed seed weights.

        - if `seed_ratings` is None, all seeds receive weight 1
        - liked seeds get positive weights
        - disliked seeds get negative weights
        - neutral seeds get weights near 0
        - if all weights are near zero, the method falls back to equal positive weights

        Parameters
        ----------
        seed_ratings : Sequence[float] or None
            Optional ratings given to the seed books by the user.

        n_seeds : int
            Number of valid seed books.

        Returns
        -------
        np.ndarray
            One weight per seed book.

        """
        if seed_ratings is None:
            return np.ones(n_seeds, dtype=np.float32)

        r = np.asarray(seed_ratings, dtype=np.float32)
        w = np.array(
            [self._centered_signed_value(x, deadzone=self.seed_deadzone) for x in r],
            dtype=np.float32,
        )

        # If all weights are near zero, fall back to equal positive seeds.
        if np.sum(np.abs(w)) < 1e-9:
            w = np.ones(n_seeds, dtype=np.float32)

        return w

    def _normalize_component_scores(self, x: np.ndarray) -> np.ndarray:
        """
        Normalize a component score vector before hybrid mixing.

        Supported modes:
        - None: no normalization
        - "maxabs": divide by the maximum absolute value
        - "zscore": subtract mean and divide by standard deviation

        Parameters
        ----------
        x : np.ndarray
            Raw component score vector aligned with the catalog.

        Returns
        -------
        np.ndarray
            Normalized score vector.
        """
        if self.component_norm is None:
            return x

        x = np.asarray(x, dtype=np.float32).copy()
        mask = np.isfinite(x) & (x != 0)

        if not np.any(mask):
            return x

        if self.component_norm == "maxabs":
            scale = float(np.max(np.abs(x[mask])))
            return x / max(scale, 1e-9)

        if self.component_norm == "zscore":
            mu = float(np.mean(x[mask]))
            sigma = float(np.std(x[mask]))
            return (x - mu) / max(sigma, 1e-9)

        raise ValueError(f"Unknown component_norm: {self.component_norm}")

    def _apply_top_k_mask(self, sim: np.ndarray, top_k: Optional[int]) -> np.ndarray:
        """
        Keep only the strongest similarities for a single seed.

        Parameters
        ----------
        sim : np.ndarray
            Similarity vector from one seed book to all catalog books.

        top_k : int or None
            Number of strongest similarities to keep.
            If None, the full vector is preserved.

        Returns
        -------
        np.ndarray
            Similarity vector where only the selected entries are kept and all
            others are set to zero.
        """
        if top_k is None or top_k >= len(sim):
            return sim

        sim = sim.copy()
        k = int(top_k)

        if self.top_k_by_abs:
            idx = np.argpartition(-np.abs(sim), k - 1)[:k]
        else:
            idx = np.argpartition(-sim, k - 1)[:k]

        out = np.zeros_like(sim)
        out[idx] = sim[idx]
        return out

    def _score_from_seed_ids(
        self,
        seed_ids: Sequence[str],
        *,
        seed_ratings: Optional[Sequence[float]] = None,
        return_components: bool = False,
        top_k_per_seed: Optional[int] = 500,
        **kwargs: Any,
    ) -> Tuple[np.ndarray, Optional[Dict[str, np.ndarray]]]:
        """
        Compute hybrid recommendation scores for all catalog items from one or
        more seed books.

        For each seed:
        - collaborative cosine similarity is computed against all items
        - content cosine similarity is computed against all items
        - both vectors may be top-k masked
        - both are weighted by the signed seed weight

        The aggregated components are divided by the sum of absolute seed weights,
        optionally normalized, and then linearly combined.

        Parameters
        ----------
        seed_ids : Sequence[str]
            Seed book IDs used to generate recommendations.

        seed_ratings : Sequence[float] or None, default=None
            Optional query-time ratings for the seed books. These are converted
            into signed seed weights.

        return_components : bool, default=False
            Whether to also return separate collaborative and content score vectors.

        top_k_per_seed : int or None, default=500
            Number of strongest similarities kept per seed and per component.
            If None, the full similarity vectors are used.

        Returns
        -------
        tuple[np.ndarray, dict[str, np.ndarray] or None]
            A pair containing:
            1. the final hybrid score vector aligned with the catalog
            2. an optional dictionary with component score vectors
        """
        self._check_is_fitted()

        valid_seed_ids, seed_positions, valid_seed_ratings = self._prepare_valid_seeds(
            seed_ids, seed_ratings
        )

        weights = self._seed_weights(valid_seed_ratings, len(valid_seed_ids))

        collab = np.zeros(len(self.catalog_), dtype=np.float32)
        content = np.zeros(len(self.catalog_), dtype=np.float32)

        for weight, pos in zip(weights, seed_positions):
            if weight == 0:
                continue

            sim_collab = linear_kernel(self.item_user_norm_[pos], self.item_user_norm_).ravel()
            sim_content = linear_kernel(self.content_norm_[pos], self.content_norm_).ravel()

            sim_collab = self._apply_top_k_mask(sim_collab, top_k_per_seed)
            sim_content = self._apply_top_k_mask(sim_content, top_k_per_seed)

            collab += weight * sim_collab.astype(np.float32)
            content += weight * sim_content.astype(np.float32)

        denom = float(np.sum(np.abs(weights))) if len(weights) else 1.0
        collab /= max(denom, 1e-9)
        content /= max(denom, 1e-9)

        collab = self._normalize_component_scores(collab)
        content = self._normalize_component_scores(content)

        final_scores = (
            self.collaborative_weight * collab
            + self.content_weight * content
        )

        components = None
        if return_components:
            components = {
                "score_collaborative": collab,
                "score_content": content,
                "score_hybrid_base": final_scores.copy(),
            }

        return final_scores, components

    def _mmr_rerank_scores(
        self,
        base_scores: np.ndarray,
        *,
        seed_ids: Sequence[str],
        n: int,
        exclude_input: bool = True,
        candidate_pool: Optional[int] = None,
        lambda_relevance: Optional[float] = None,
    ) -> np.ndarray:
        """
        Greedy MMR reranking using content similarity as the redundancy term.

        MMR selects items greedily. At each step it chooses the candidate with the
        best tradeoff between:
        - relevance according to the base score
        - novelty with respect to already selected items

        Redundancy is measured using content cosine similarity

        High lambda -> more relevance
        Low lambda -> more diversity
        """
        scores = np.asarray(base_scores, dtype=np.float64).copy()

        if exclude_input:
            for pos in self._catalog_positions(seed_ids):
                scores[pos] = -np.inf

        order = np.argsort(-scores)
        order = order[np.isfinite(scores[order])]

        pool_size = candidate_pool or self.rerank_pool
        pool = order[:pool_size].tolist()
        if not pool:
            return scores

        lam = self.diversity_lambda if lambda_relevance is None else lambda_relevance
        if lam is None:
            return scores

        rel = scores[pool].copy()
        rel = rel / max(np.max(np.abs(rel)), 1e-9)
        rel_map = {pos: float(val) for pos, val in zip(pool, rel)}

        selected: List[int] = []

        while pool and len(selected) < n:
            best_pos = None
            best_val = -np.inf

            for cand in pool:
                if not selected:
                    redundancy = 0.0
                else:
                    redundancy = float(
                        linear_kernel(
                            self.content_norm_[cand],
                            self.content_norm_[selected],
                        ).max()
                    )

                mmr_val = lam * rel_map[cand] - (1.0 - lam) * redundancy

                if mmr_val > best_val:
                    best_val = mmr_val
                    best_pos = cand

            selected.append(best_pos)
            pool.remove(best_pos)

        reranked = np.full_like(scores, -np.inf, dtype=np.float64)

        for rank, pos in enumerate(selected):
            reranked[pos] = float(len(selected) - rank) + 1e-3 * rel_map[pos]

        return reranked

    def recommend_by_ids(
        self,
        seed_book_ids: ArrayLikeId,
        *,
        n: int = 10,
        seed_ratings: Optional[Sequence[float]] = None,
        exclude_input: bool = True,
        return_components: bool = False,
        rerank_diversity: bool = False,
        candidate_pool: Optional[int] = None,
        mmr_lambda: Optional[float] = None,
        **kwargs: Any,
    ) -> pd.DataFrame:
        """
        Recommend books from one or more seed book IDs.

        Parameters
        ----------
        seed_book_ids : ArrayLikeId
            One seed ID or a sequence of seed IDs.

        n : int, default=10
            Number of recommendations to return.

        seed_ratings : Sequence[float] or None, default=None
            Optional query-time ratings for the seed books.

        exclude_input : bool, default=True
            Whether to exclude the seed books themselves from the output.

        return_components : bool, default=False
            Whether to include separate score components in the result table.

        rerank_diversity : bool, default=False
            Whether to apply MMR reranking on top of the hybrid base score.

        candidate_pool : int or None, default=None
            Number of top hybrid candidates considered for diversity reranking.
            If None, `self.rerank_pool` is used.

        mmr_lambda : float or None, default=None
            Relevance-diversity tradeoff used by MMR reranking.
            If None, `self.diversity_lambda` is used.

        **kwargs : Any
            Additional keyword arguments forwarded to `_score_from_seed_ids`,
            for example `top_k_per_seed`.

        Returns
        -------
        pd.DataFrame
            Final recommendation table with metadata and score columns.

        """
        self._check_is_fitted()
        seed_ids = self._normalize_seed_ids(seed_book_ids)

        scores, components = self._score_from_seed_ids(
            seed_ids,
            seed_ratings=seed_ratings,
            return_components=return_components,
            **kwargs,
        )

        if rerank_diversity:
            base_scores = scores.copy()
            scores = self._mmr_rerank_scores(
                base_scores,
                seed_ids=seed_ids,
                n=n,
                exclude_input=exclude_input,
                candidate_pool=candidate_pool,
                lambda_relevance=mmr_lambda,
            )
            if return_components:
                components = {} if components is None else dict(components)
                components["score_reranked"] = scores
                components["score_hybrid_base"] = base_scores

        return self._build_output(
            scores,
            seed_ids=seed_ids,
            n=n,
            exclude_input=exclude_input,
            component_scores=components if return_components else None,
        )

## User-based KNN recommender

`UserKNNRecommender` is a **collaborative filtering** method based on **user-user similarity**.

Instead of learning latent factors or using book metadata, it recommends books by finding **historical users whose rating behavior is similar to the current query**.

In this implementation, the query is represented as a **temporary user** built from the input seed books.

---

### Main idea

The method works in three steps:

1. build a temporary query user from the seed books
2. find historical users most similar to that query
3. predict book ratings from the ratings of those similar users

So the recommendation is driven by the idea:

> users who rated the seed books similarly may like simmilar books as query user

---

### 1. Explicit interaction matrix

The model works only with **explicit ratings**.

After cleaning the interaction table, it builds a user-item rating matrix:

$$
R \in \mathbb{R}^{U \times I}
$$

where:

- $U$ is the number of users
- $I$ is the number of books
- $R_{u,i}$ is the rating of user $u$ for item $i$

Only users and books with enough ratings are kept so that similarity estimates are more reliable.

---

### 2. Mean-centering user ratings

Different users use rating scales differently.
Some rate generously, others more strictly.

To reduce this bias, each user's ratings are centered by subtracting that user's average rating:

$$
\mu_u = \frac{1}{|I_u|} \sum_{i \in I_u} R_{u,i}
$$

$$
R^{(c)}_{u,i} = R_{u,i} - \mu_u
$$

where:

- $\mu_u$ is the mean rating of user $u$
- $I_u$ is the set of items rated by user $u$

This means similarity is based more on **relative preference pattern** than on absolute rating level.

---

### 3. Query user representation

At recommendation time, the seed books are converted into a temporary query profile.

If no `seed_ratings` are provided, each seed book receives the default value:

$$
r^{(q)}_m = \text{like\_rating}
$$

If seed ratings are provided, those values are used directly.

The query mean is:

$$
\mu_q = \frac{1}{M} \sum_{m=1}^{M} r^{(q)}_m
$$

where $M$ is the number of seed books used in the query.

The implementation then builds centered query values as:

$$
q_m = r^{(q)}_m - \frac{\mu_q + \mu_{\text{global}}}{2}
$$

where:

- $\mu_q$ is the mean of the query ratings
- $\mu_{\text{global}}$ is the global average rating in the training data

This gives the temporary query user a centered representation that can be compared with the centered historical users.

---

### 4. User-user similarity

The recommender compares the temporary query user with all historical users.


The query is compared with all centered user vectors:

$$
\text{sim}(q,u) =
\frac{q \cdot R^{(c)}_u}
{\|q\| \, \|R^{(c)}_u\|}
$$

---

### 5. Neighbor selection

After similarities are computed:

- optionally, negative similarities are replaced by zero
- only the top `k_neighbors` users are kept
- only neighbors with positive similarity contribute to prediction

This means the method usually relies only on the most relevant similar users.

---

### 6. Rating prediction

For each candidate item $j$, the predicted score is computed from the selected neighbors.

First, the weighted centered contribution is:

$$
\text{num}_j = \sum_{u \in N(q)} \text{sim}(q,u)\, R^{(c)}_{u,j}
$$

Then the normalization term is:

$$
\text{den}_j = \sum_{u \in N(q),\, u \text{ rated } j} |\text{sim}(q,u)|
$$

The final prediction is:

$$
\hat{r}_{q,j} = \mu_q + \frac{\text{num}_j}{\text{den}_j}
$$

So the predicted rating is the query user's mean preference plus a weighted deviation estimated from similar users.

Predictions are then clipped to the configured rating range:

$$
\hat{r}_{q,j} \in [\text{clip\_min}, \text{clip\_max}]
$$

---

### Practical meaning of the main parameters

- `k_neighbors` controls how many similar users are used for prediction
- `like_rating` is the default rating assigned to seed books when no seed ratings are given
- `zero_negative_sims=True` keeps only positively similar users
- `similarity_mode="full"` compares users in the full rating space
- `similarity_mode="query_items"` compares users only on the seed-book columns
- `neighbors_only=True` returns only items supported by selected neighbors

In [5]:
class UserKNNRecommender(SeedBookRecommenderBase):
    """
    User-based k-nearest-neighbors recommender for explicit feedback.

    The model represents each historical user by a vector of explicit ratings,
    mean-centers those ratings per user, and compares a temporary query user
    built from the seed books against the historical users using cosine similarity.

    Recommendations are produced by aggregating item-specific rating deviations
    from the most similar users.

    Parameters
    ----------
    user_col : str, default="user_id"
        Name of the column containing user identifiers in the ratings table.

    item_col : str, default="book_id"
        Name of the column containing item or book identifiers in the ratings table.

    rating_col : str, default="book_rating"
        Name of the column containing explicit rating values.

    explicit_min : float, default=1.0
        Minimum explicit rating value to keep when preparing the interaction table.

    min_user_ratings : int, default=20
        Minimum number of ratings required for a user to remain in the training set.

    min_item_ratings : int, default=20
        Minimum number of ratings required for an item to remain in the training set.

    k_neighbors : int, default=50
        Number of nearest users retained after similarity computation.

    like_rating : float, default=9.0
        Default rating assigned to each seed book if `seed_ratings` are not provided.

    clip_min : float, default=1.0
        Minimum allowed predicted rating.

    clip_max : float, default=10.0
        Maximum allowed predicted rating.

    zero_negative_sims : bool, default=True
        If True, negative user-query similarities are set to zero so that only
        positively similar users contribute.

    similarity_mode : {"full", "query_items"}, default="full"
        Strategy used for user-query similarity:
        - "full": compare in the full centered item space
        - "query_items": compare only on the seed-item columns

    neighbors_only : bool, default=True
        If True, only items supported by selected neighbors receive predictions.
        If False, unsupported items fall back to the query mean.
    """
    def __init__(
        self,
        *,
        user_col: str = "user_id",
        item_col: str = "book_id",
        rating_col: str = "book_rating",
        explicit_min: float = 1.0,
        min_user_ratings: int = 20,
        min_item_ratings: int = 20,
        k_neighbors: int = 50,
        like_rating: float = 9.0,
        clip_min: float = 1.0,
        clip_max: float = 10.0,
        zero_negative_sims: bool = True,
        similarity_mode: str = "full",
        neighbors_only: bool = True,
        **kwargs: Any,
    ):
        super().__init__(**kwargs)
        self.user_col = user_col
        self.item_col = item_col
        self.rating_col = rating_col

        self.explicit_min = explicit_min
        self.min_user_ratings = min_user_ratings
        self.min_item_ratings = min_item_ratings
        self.k_neighbors = k_neighbors
        self.like_rating = like_rating
        self.clip_min = clip_min
        self.clip_max = clip_max
        self.zero_negative_sims = zero_negative_sims

        if similarity_mode not in {"full", "query_items"}:
            raise ValueError("similarity_mode must be either 'full' or 'query_items'.")
        self.similarity_mode = similarity_mode
        self.neighbors_only = neighbors_only

        self.train_df_: Optional[pd.DataFrame] = None
        self.user2idx_: Optional[Dict[str, int]] = None
        self.item2idx_: Optional[Dict[str, int]] = None
        self.idx2item_: Optional[np.ndarray] = None

        self.R_raw_: Optional[csr_matrix] = None
        self.R_centered_raw_: Optional[csr_matrix] = None
        self.R_centered_norm_: Optional[csr_matrix] = None
        self.user_mean_: Optional[np.ndarray] = None
        self.global_mean_: float = 0.0

    def fit(self, books: pd.DataFrame, ratings: pd.DataFrame) -> "UserKNNRecommender":
        """
        Fit the user-based KNN recommender on the explicit rating table.

        The method:
        1. builds the shared metadata catalog
        2. cleans the explicit interaction table
        3. builds a sparse user-item rating matrix
        4. computes per-user mean ratings
        5. mean-centers user ratings
        6. stores row-normalized centered user vectors for cosine similarity

        Parameters
        ----------
        books : pd.DataFrame
            Book metadata table used to build the shared catalog.

        ratings : pd.DataFrame
            Explicit user-item-rating table used for collaborative filtering.

        Returns
        -------
        UserKNNRecommender
            The fitted recommender instance.
        """
        self._set_catalog(books)

        train_df = prepare_explicit_interactions(
            ratings,
            user_col=self.user_col,
            item_col=self.item_col,
            rating_col=self.rating_col,
            explicit_min=self.explicit_min,
            min_user_ratings=self.min_user_ratings,
            min_item_ratings=self.min_item_ratings,
        )

        valid_ids = set(self.catalog_[self.id_col].astype(str))
        train_df = train_df[train_df[self.item_col].isin(valid_ids)].copy()
        if train_df.empty:
            raise ValueError("No explicit interactions remain after cleaning.")

        user_ids = np.sort(train_df[self.user_col].astype(str).unique())
        item_ids = np.sort(train_df[self.item_col].astype(str).unique())

        self.user2idx_ = {u: idx for idx, u in enumerate(user_ids)}
        self.item2idx_ = {it: idx for idx, it in enumerate(item_ids)}
        self.idx2item_ = item_ids
        self.train_df_ = train_df

        u_idx = train_df[self.user_col].map(self.user2idx_).to_numpy(np.int64)
        i_idx = train_df[self.item_col].map(self.item2idx_).to_numpy(np.int64)
        r = train_df[self.rating_col].to_numpy(np.float32)

        self.R_raw_ = csr_matrix(
            (r, (u_idx, i_idx)),
            shape=(len(user_ids), len(item_ids)),
            dtype=np.float32,
        )

        row_sum = np.asarray(self.R_raw_.sum(axis=1)).ravel()
        row_cnt = np.asarray((self.R_raw_ != 0).sum(axis=1)).ravel()
        self.user_mean_ = np.divide(row_sum, row_cnt, out=np.zeros_like(row_sum), where=row_cnt != 0)
        self.global_mean_ = float(r.mean()) if len(r) else 0.0

        self.R_centered_raw_ = self.R_raw_.copy().tocsr()
        row_for_data = np.repeat(np.arange(self.R_centered_raw_.shape[0]), np.diff(self.R_centered_raw_.indptr))
        self.R_centered_raw_.data = self.R_centered_raw_.data - self.user_mean_[row_for_data]

        self.R_centered_norm_ = normalize(self.R_centered_raw_.copy(), norm="l2", axis=1)
        return self

    def _prepare_query(
        self,
        seed_ids: Sequence[str],
        seed_ratings: Optional[Sequence[float]],
    ) -> Tuple[np.ndarray, np.ndarray, float]:
        """
        Convert the seed books into a temporary query-user representation.

        Only seed books that exist in the explicit training item space are kept.
        If no explicit seed ratings are provided, each seed receives the default
        value `like_rating`.

        Parameters
        ----------
        seed_ids : Sequence[str]
            Normalized seed book IDs.

        seed_ratings : Sequence[float] or None
            Optional ratings associated with the seed books.

        Returns
        -------
        tuple[np.ndarray, np.ndarray, float]
            A tuple containing:
            1. item indices of valid seed books
            2. query ratings aligned with those item indices
            3. mean rating of the query profile
        """
        if self.item2idx_ is None:
            raise RuntimeError("Call fit(...) first.")

        idxs = []
        ratings = []

        if seed_ratings is not None and len(seed_ratings) != len(seed_ids):
            raise ValueError("seed_ratings must have the same length as seed_book_ids.")

        for i, book_id in enumerate(seed_ids):
            item_idx = self.item2idx_.get(book_id)
            if item_idx is None:
                continue
            idxs.append(item_idx)
            ratings.append(float(self.like_rating) if seed_ratings is None else float(seed_ratings[i]))

        if not idxs:
            raise KeyError("None of the provided seed book IDs exist in the explicit training matrix.")

        idxs = np.asarray(idxs, dtype=np.int64)
        ratings = np.asarray(ratings, dtype=np.float32)
        query_mean = float(np.mean(ratings)) if len(ratings) else self.global_mean_
        return idxs, ratings, query_mean


    def _score_from_seed_ids(
        self,
        seed_ids: Sequence[str],
        *,
        seed_ratings: Optional[Sequence[float]] = None,
        return_components: bool = False,
        **kwargs: Any,
    ) -> Tuple[np.ndarray, Optional[Dict[str, np.ndarray]]]:
        """"
        Predict scores for all catalog items from a temporary query user.

        The method:
        1. builds the temporary query profile from the seed books
        2. computes similarity between the query and historical users
        3. keeps the strongest neighbors
        4. predicts item ratings from neighbor deviations
        5. maps those predictions back to the full catalog

        Parameters
        ----------
        seed_ids : Sequence[str]
            Seed book IDs used to define the query.

        seed_ratings : Sequence[float] or None, default=None
            Optional ratings for the seed books. If omitted, all seed books
            receive the default value `like_rating`.

        return_components : bool, default=False
            Whether to also return component score arrays.

        Returns
        -------
        tuple[np.ndarray, dict[str, np.ndarray] or None]
            A pair containing:
            1. score vector aligned with the full catalog
            2. optional component dictionary

        """
        self._check_is_fitted()

        seed_item_idxs, query_ratings, query_mean = self._prepare_query(seed_ids, seed_ratings)
        q_values = query_ratings - ((query_mean + self.global_mean_) / 2)

        if self.similarity_mode == "full":
            q = csr_matrix(
                (q_values, (np.zeros(len(seed_item_idxs)), seed_item_idxs)),
                shape=(1, self.R_centered_norm_.shape[1]),
                dtype=np.float32,
            )
            q = normalize(q, norm="l2", axis=1)
            sims = linear_kernel(q, self.R_centered_norm_).ravel()

        elif self.similarity_mode == "query_items":
            R_seed = self.R_centered_raw_[:, seed_item_idxs]
            R_seed_norm = normalize(R_seed.copy(), norm="l2", axis=1)

            q_seed = csr_matrix(
                (q_values, (np.zeros(len(seed_item_idxs)), np.arange(len(seed_item_idxs)))),
                shape=(1, len(seed_item_idxs)),
                dtype=np.float32,
            )
            q_seed = normalize(q_seed, norm="l2", axis=1)
            sims = linear_kernel(q_seed, R_seed_norm).ravel()

        else:
            raise ValueError("similarity_mode must be either 'full' or 'query_items'.")

        if self.zero_negative_sims:
            sims = np.maximum(sims, 0.0)

        if self.k_neighbors is not None and self.k_neighbors < len(sims):
            top_u = np.argpartition(-sims, self.k_neighbors)[: self.k_neighbors]
            sims_masked = np.zeros_like(sims)
            sims_masked[top_u] = sims[top_u]
            sims = sims_masked

        neighbor_mask = sims > 0
        neighbor_idxs = np.flatnonzero(neighbor_mask)

        n_items = len(self.idx2item_)
        pred = np.full(n_items, np.nan, dtype=np.float32)

        if len(neighbor_idxs) > 0:
            sims_nbr = sims[neighbor_idxs]

            Rc_nbr = self.R_centered_raw_[neighbor_idxs]
            num = np.asarray(Rc_nbr.T.dot(sims_nbr)).ravel()

            support = self.R_raw_[neighbor_idxs].copy().astype(np.float32)
            support.data = np.ones_like(support.data, dtype=np.float32)
            den = np.asarray(support.T.dot(np.abs(sims_nbr))).ravel()

            supported = den > 1e-9
            pred[supported] = query_mean + (num[supported] / den[supported])
            pred[supported] = np.clip(pred[supported], self.clip_min, self.clip_max)

            if not self.neighbors_only:
                pred[~supported] = np.clip(query_mean, self.clip_min, self.clip_max)

        scores = np.full(len(self.catalog_), np.nan, dtype=np.float64)
        for item_idx, book_id in enumerate(self.idx2item_):
            catalog_pos = self.catalog_id_to_pos_.get(str(book_id))
            if catalog_pos is not None:
                scores[catalog_pos] = pred[item_idx]

        components = None
        if return_components:
            components = {
                "predicted_rating": scores,
            }
        return scores, components


## Biased matrix factorization recommender

This part implements a **biased matrix factorization** recommender for **explicit ratings**.

The method learns a compact latent representation of users and books from the rating matrix.
Instead of representing each user or book directly by all ratings, it represents them by a smaller number of latent factors.

The fitted model predicts a rating as:

$$
\hat{r}_{u,i} = \mu + b_u + b_i + p_u^\top q_i
$$

where:

- $\mu$ is the global mean rating
- $b_u$ is the user bias
- $b_i$ is the item bias
- $p_u \in \mathbb{R}^k$ is the latent vector of user $u$
- $q_i \in \mathbb{R}^k$ is the latent vector of item $i$

So the prediction has two parts:

1. **bias terms**, which capture general tendencies such as "this user rates high" or "this book is broadly popular"
2. **latent interaction term**, which captures hidden preference structure

---

### 1. Training objective

The model is trained from explicit ratings by minimizing the regularized squared error over observed ratings:

$$
\mathcal{L} =
\sum_{(u,i)\in\Omega}
\left(r_{u,i} - \mu - b_u - b_i - p_u^\top q_i\right)^2
+
\lambda \left(
\sum_u \|p_u\|^2 +
\sum_i \|q_i\|^2 +
\sum_u b_u^2 +
\sum_i b_i^2
\right)
$$

where:

- $\Omega$ is the set of observed user-item ratings
- $\lambda$ is the regularization strength

The implementation optimizes this objective with **stochastic gradient descent (SGD)**.

---

### 2. SGD update logic

For one observed rating $(u,i,r_{u,i})$, the current prediction is:

$$
\hat{r}_{u,i} = \mu + b_u + b_i + p_u^\top q_i
$$

and the prediction error is:

$$
e_{u,i} = r_{u,i} - \hat{r}_{u,i}
$$

Then the parameters are updated as:

$$
b_u \leftarrow b_u + \eta (e_{u,i} - \lambda b_u)
$$

$$
b_i \leftarrow b_i + \eta (e_{u,i} - \lambda b_i)
$$

$$
p_u \leftarrow p_u + \eta (e_{u,i} q_i - \lambda p_u)
$$

$$
q_i \leftarrow q_i + \eta (e_{u,i} p_u - \lambda q_i)
$$

where:

- $\eta$ is the learning rate
- $\lambda$ is the regularization strength

This is repeated for many epochs over shuffled training examples.

---

### 3. Why matrix factorization is useful

Unlike memory-based KNN methods, matrix factorization does not rely directly on local rating overlap.

Instead, it learns **latent dimensions** that can capture hidden structure such as:

- preference for genre or style
- popularity versus niche taste
- broad user tendencies in rating behavior

Because of that, it often generalizes better than neighborhood methods when the rating matrix is sparse.

---

### 4. Recommendation from seed books

At recommendation time, the user is not necessarily a known training user.
So instead of looking up an existing user vector, the recommender estimates a **temporary user profile** from the supplied seed books.

The temporary user follows the model:

$$
r(i) \approx \mu + b_i + b_{\text{temp}} + q_i^\top p_{\text{temp}}
$$

where:

- $b_{\text{temp}}$ is the bias of the temporary user
- $p_{\text{temp}}$ is the latent vector of the temporary user
- $q_i$ and $b_i$ are the learned item parameters

If no explicit `seed_ratings` are given, each seed book receives the default value `like_rating`.

---

### 5. Fitting the temporary user

For the seed books, the recommender solves a small **ridge regression** problem.

First define the target values:

$$
y_m = r_m - (\mu + b_{i_m})
$$

for each seed item $i_m$ with rating $r_m$.

Then form the design matrix:

$$
A =
\begin{bmatrix}
1 & q_{i_1}^\top \\
1 & q_{i_2}^\top \\
\vdots & \vdots \\
1 & q_{i_M}^\top
\end{bmatrix}
$$

and the temporary user parameter vector:

$$
\beta =
\begin{bmatrix}
b_{\text{temp}} \\
p_{\text{temp}}
\end{bmatrix}
$$

The model solves:

$$
\min_\beta \|A\beta - y\|^2 + \beta^\top \Lambda \beta
$$

where the regularization matrix is diagonal:

$$
\Lambda =
\operatorname{diag}(\lambda_b, \lambda_p, \dots, \lambda_p)
$$

Here:

- $\lambda_b$ corresponds to `bias_ridge`
- $\lambda_p$ corresponds to `user_ridge`

The closed-form ridge solution is:

$$
\beta = (A^\top A + \Lambda)^{-1} A^\top y
$$

This gives the temporary user bias and temporary latent factors.

---

### 6. Final prediction for all books

Once the temporary user is estimated, the predicted rating for every item is:

$$
\hat{r}_i = \mu + b_{\text{temp}} + b_i + q_i^\top p_{\text{temp}}
$$

The predictions are then clipped into the valid rating range:

$$
\hat{r}_i \in [\text{clip\_min}, \text{clip\_max}]
$$

These predicted ratings are used as recommendation scores.

---

### Practical meaning of the main parameters

- `k_factors` controls the dimensionality of the latent space
- `lr` controls the SGD step size
- `reg` controls regularization during matrix factorization training
- `n_epochs` controls how many full passes over the training data are made
- `like_rating` is the default seed rating if no explicit query ratings are provided
- `user_ridge` regularizes the temporary user latent vector
- `bias_ridge` regularizes the temporary user bias
- `clip_min` and `clip_max` keep predicted ratings in the valid rating range

---

This method learns hidden user and item preference factors from explicit ratings, then fits a temporary latent user from the input seed books and predicts which other books that temporary user would rate highly.

In [6]:
def train_biased_mf_sgd(
    u_idx: np.ndarray,
    i_idx: np.ndarray,
    r: np.ndarray,
    n_users: int,
    n_items: int,
    *,
    k: int = 50,
    lr: float = 0.01,
    reg: float = 0.05,
    n_epochs: int = 20,
    seed: int = 42,
) -> Tuple[float, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Train a biased matrix factorization model with stochastic gradient descent.

    The fitted model predicts ratings as

        r_hat(u, i) = mu + bu[u] + bi[i] + P[u] @ Q[i]

    where:
    - mu is the global mean rating
    - bu and bi are user and item bias terms
    - P and Q are latent-factor matrices

    Parameters
    ----------
    u_idx : np.ndarray
        Integer user indices for the observed training ratings.

    i_idx : np.ndarray
        Integer item indices for the observed training ratings.

    r : np.ndarray
        Observed rating values aligned with `u_idx` and `i_idx`.

    n_users : int
        Total number of users in the training matrix.

    n_items : int
        Total number of items in the training matrix.

    k : int, default=50
        Number of latent factors.

    lr : float, default=0.01
        Learning rate used by stochastic gradient descent.

    reg : float, default=0.05
        L2 regularization strength applied to biases and latent factors.

    n_epochs : int, default=20
        Number of full passes over the observed ratings.

    seed : int, default=42
        Random seed used for initialization and shuffling.

    Returns
    -------
    tuple[float, np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        A tuple containing:
        1. global mean rating `mu`
        2. user bias vector `bu`
        3. item bias vector `bi`
        4. user latent matrix `P`
        5. item latent matrix `Q`
    """

    rng = np.random.default_rng(seed)
    mu = float(np.mean(r)) if len(r) else 0.0

    bu = np.zeros(n_users, dtype=np.float32)
    bi = np.zeros(n_items, dtype=np.float32)
    P = (0.1 * rng.standard_normal((n_users, k))).astype(np.float32)
    Q = (0.1 * rng.standard_normal((n_items, k))).astype(np.float32)

    order = np.arange(len(r))
    for _ in range(n_epochs):
        rng.shuffle(order)
        for t in order:
            u = u_idx[t]
            it = i_idx[t]
            rt = r[t]

            pred = mu + bu[u] + bi[it] + P[u].dot(Q[it])
            err = rt - pred

            bu[u] += lr * (err - reg * bu[u])
            bi[it] += lr * (err - reg * bi[it])

            Pu = P[u].copy()
            P[u] += lr * (err * Q[it] - reg * P[u])
            Q[it] += lr * (err * Pu - reg * Q[it])

    return mu, bu, bi, P, Q


class BiasedMFRecommender(SeedBookRecommenderBase):
    """
    Biased matrix factorization recommender for explicit feedback.

    The model first learns item and user parameters from historical ratings:

        r_hat(u, i) = mu + bu[u] + bi[i] + P[u] @ Q[i]

    The final recommendation scores are predicted ratings for all items.

    Parameters
    ----------
    user_col : str, default="user_id"
        Name of the column containing user identifiers in the ratings table.

    item_col : str, default="book_id"
        Name of the column containing item or book identifiers in the ratings table.

    rating_col : str, default="book_rating"
        Name of the column containing explicit rating values.

    explicit_min : float, default=1.0
        Minimum explicit rating retained during interaction preprocessing.

    min_user_ratings : int, default=20
        Minimum number of ratings required for a user to remain in the training set.

    min_item_ratings : int, default=20
        Minimum number of ratings required for an item to remain in the training set.

    k_factors : int, default=50
        Number of latent factors used by the matrix factorization model.

    lr : float, default=0.01
        Learning rate for SGD training.

    reg : float, default=0.05
        Regularization strength for SGD training.

    n_epochs : int, default=20
        Number of SGD training epochs.

    like_rating : float, default=9.0
        Default rating assigned to each seed book if `seed_ratings` are not supplied.

    clip_min : float, default=1.0
        Minimum allowed predicted rating.

    clip_max : float, default=10.0
        Maximum allowed predicted rating.

    user_ridge : float, default=0.10
        Ridge regularization strength for the temporary user's latent factor vector.

    bias_ridge : float, default=0.01
        Ridge regularization strength for the temporary user's bias term.

    seed : int, default=42
        Random seed for reproducible matrix factorization training.
    """

    def __init__(
        self,
        *,
        user_col: str = "user_id",
        item_col: str = "book_id",
        rating_col: str = "book_rating",
        explicit_min: float = 1.0,
        min_user_ratings: int = 20,
        min_item_ratings: int = 20,
        k_factors: int = 50,
        lr: float = 0.01,
        reg: float = 0.05,
        n_epochs: int = 20,
        like_rating: float = 9.0,
        clip_min: float = 1.0,
        clip_max: float = 10.0,
        user_ridge: float = 0.10,
        bias_ridge: float = 0.01,
        seed: int = 42,
        **kwargs: Any,
    ):
        super().__init__(**kwargs)
        self.user_col = user_col
        self.item_col = item_col
        self.rating_col = rating_col

        self.explicit_min = explicit_min
        self.min_user_ratings = min_user_ratings
        self.min_item_ratings = min_item_ratings

        self.k_factors = k_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.like_rating = like_rating
        self.clip_min = clip_min
        self.clip_max = clip_max

        self.user_ridge = user_ridge
        self.bias_ridge = bias_ridge   # NEW
        self.seed = seed               # NEW

        self.item2idx_: Optional[Dict[str, int]] = None
        self.idx2item_: Optional[np.ndarray] = None
        self.mu_: float = 0.0
        self.bu_: Optional[np.ndarray] = None
        self.bi_: Optional[np.ndarray] = None
        self.P_: Optional[np.ndarray] = None
        self.Q_: Optional[np.ndarray] = None

    def fit(self, books: pd.DataFrame, ratings: pd.DataFrame) -> "BiasedMFRecommender":
        """
        Fit the biased matrix factorization model on explicit ratings.

        The method:
        1. builds the shared metadata catalog
        2. cleans the explicit interaction table
        3. converts user and item IDs to integer indices
        4. trains the biased matrix factorization model with SGD
        5. stores the learned item and user parameters

        Parameters
        ----------
        books : pd.DataFrame
            Book metadata table used to build the shared catalog.

        ratings : pd.DataFrame
            Explicit user-item-rating table used for model training.

        Returns
        -------
        BiasedMFRecommender
            The fitted recommender instance.
        """
        self._set_catalog(books)

        train_df = prepare_explicit_interactions(
            ratings,
            user_col=self.user_col,
            item_col=self.item_col,
            rating_col=self.rating_col,
            explicit_min=self.explicit_min,
            min_user_ratings=self.min_user_ratings,
            min_item_ratings=self.min_item_ratings,
        )

        valid_ids = set(self.catalog_[self.id_col].astype(str))
        train_df = train_df[train_df[self.item_col].isin(valid_ids)].copy()
        if train_df.empty:
            raise ValueError("No explicit interactions remain after cleaning.")

        user_ids = np.sort(train_df[self.user_col].astype(str).unique())
        item_ids = np.sort(train_df[self.item_col].astype(str).unique())

        user2idx = {u: idx for idx, u in enumerate(user_ids)}
        self.item2idx_ = {it: idx for idx, it in enumerate(item_ids)}
        self.idx2item_ = item_ids

        u_idx = train_df[self.user_col].map(user2idx).to_numpy(np.int64)
        i_idx = train_df[self.item_col].map(self.item2idx_).to_numpy(np.int64)
        r = train_df[self.rating_col].to_numpy(np.float32)

        self.mu_, self.bu_, self.bi_, self.P_, self.Q_ = train_biased_mf_sgd(
            u_idx,
            i_idx,
            r,
            n_users=len(user_ids),
            n_items=len(item_ids),
            k=self.k_factors,
            lr=self.lr,
            reg=self.reg,
            n_epochs=self.n_epochs,
            seed=self.seed,
        )

        return self

    def _fit_temporary_user(
        self,
        item_indices: np.ndarray,
        ratings: np.ndarray,
    ) -> Tuple[float, np.ndarray]:
        """
        Estimate a temporary user's bias and latent factors from seed books.

        The temporary user is fitted with ridge regression under the model

            rating(i) ≈ mu + bi[i] + b_user + Q[i] @ p_user

        where:
        - mu is the global mean
        - bi[i] is the learned item bias
        - Q[i] is the learned item latent vector
        - b_user and p_user are the temporary user parameters to estimate

        Parameters
        ----------
        item_indices : np.ndarray
            Integer indices of seed items in the MF item space.

        ratings : np.ndarray
            Ratings associated with the seed items.

        Returns
        -------
        tuple[float, np.ndarray]
            A tuple containing:
            1. the temporary user bias
            2. the temporary user latent factor vector
        """

        Q = self.Q_[item_indices].astype(np.float64)
        y = ratings.astype(np.float64) - (
            float(self.mu_) + self.bi_[item_indices].astype(np.float64)
        )

        m = Q.shape[0]
        A = np.concatenate(
            [np.ones((m, 1), dtype=np.float64), Q],
            axis=1,
        )

        # Separate regularization for bias and latent factors
        reg_vec = np.concatenate(
            [
                np.array([self.bias_ridge], dtype=np.float64),
                np.full(Q.shape[1], self.user_ridge, dtype=np.float64),
            ]
        )

        AtA = A.T @ A + np.diag(reg_vec)
        Aty = A.T @ y

        beta = np.linalg.solve(AtA, Aty)

        b_user = float(beta[0])
        p_user = beta[1:].astype(np.float32)
        return b_user, p_user

    def _score_from_seed_ids(
        self,
        seed_ids: Sequence[str],
        *,
        seed_ratings: Optional[Sequence[float]] = None,
        return_components: bool = False,
        **kwargs: Any,
    ) -> Tuple[np.ndarray, Optional[Dict[str, np.ndarray]]]:
        """
        Predict scores for all catalog items using the temporary MF user.

        The method:
        1. keeps only seed books present in the MF item space
        2. assigns ratings to those seed books
        3. fits a temporary user bias and latent vector
        4. predicts ratings for all trained items
        5. maps predictions back to the full catalog

        Parameters
        ----------
        seed_ids : Sequence[str]
            Seed book IDs used to define the temporary user.

        seed_ratings : Sequence[float] or None, default=None
            Optional ratings for the seed books. If omitted, each seed book is
            assigned the default value `like_rating`.

        return_components : bool, default=False
            Whether to also return component score arrays.

        Returns
        -------
        tuple[np.ndarray, dict[str, np.ndarray] or None]
            A pair containing:
            1. score vector aligned with the full catalog
            2. optional component dictionary
        """
        self._check_is_fitted()

        if seed_ratings is not None and len(seed_ratings) != len(seed_ids):
            raise ValueError("seed_ratings must have the same length as seed_book_ids.")

        item_idxs = []
        ratings = []
        for i, book_id in enumerate(seed_ids):
            item_idx = self.item2idx_.get(book_id)
            if item_idx is None:
                continue
            item_idxs.append(item_idx)
            ratings.append(float(self.like_rating) if seed_ratings is None else float(seed_ratings[i]))

        if not item_idxs:
            raise KeyError("None of the provided seed book IDs exist in the MF training matrix.")

        item_idxs = np.asarray(item_idxs, dtype=np.int64)
        ratings = np.asarray(ratings, dtype=np.float32)

        b_user, p_user = self._fit_temporary_user(item_idxs, ratings)

        pred = self.mu_ + b_user + self.bi_ + self.Q_.dot(p_user)
        pred = np.clip(pred, self.clip_min, self.clip_max)

        scores = np.full(len(self.catalog_), np.nan, dtype=np.float64)
        for item_idx, book_id in enumerate(self.idx2item_):
            catalog_pos = self.catalog_id_to_pos_.get(str(book_id))
            if catalog_pos is not None:
                scores[catalog_pos] = float(pred[item_idx])

        components = None
        if return_components:
            components = {"predicted_rating": scores}
        return scores, components


## Catalog inspection

This inspection step is useful before fitting the models.

It lets you quickly verify:

- how many books remain in the catalog
- how many explicit interactions remain after filtering
- whether key metadata columns such as popularity and average rating look correct


In [7]:
catalog_preview = build_book_catalog(books)
explicit_preview = prepare_explicit_interactions(
    ratings,
    explicit_min=1.0,
    min_user_ratings=20,
    min_item_ratings=20,
)

print(f"Catalog size: {len(catalog_preview):,}")
print(f"Explicit interactions after filtering: {len(explicit_preview):,}")

display(
    catalog_preview[[c for c in ["book_id", "title", "author", "explicit_ratings", "avg_explicit_rating"] if c in catalog_preview.columns]]
    .sort_values("explicit_ratings", ascending=False).head(10)
)


Catalog size: 248,108
Explicit interactions after filtering: 54,372


,book_id,title,author,explicit_ratings,avg_explicit_rating
0,203104,The Lovely Bones: A Novel,Alice Sebold,707,8.185290
1,242026,Wild Animus,Rich Shapero,581,4.390706
2,189376,The Da Vinci Code,Dan Brown,487,8.435318
3,211699,The Red Tent (Bestselling Backlist),Anita Diamant,383,8.182768
4,54522,Divine Secrets of the Ya-Ya Sisterhood: A Novel,Rebecca Wells,320,7.887500
5,82717,Harry Potter and the Sorcerer's Stone (Harry P...,J. K. Rowling,313,8.939297
6,213912,The Secret Life of Bees,Sue Monk Kidd,307,8.452769
7,240288,Where the Heart Is (Oprah's Book Club (Paperba...,Billie Letts,295,8.142373
8,5716,A Painted House,John Grisham,281,7.338078
9,77024,Girl with a Pearl Earring,Tracy Chevalier,278,7.982014


## Fit the recommenders

The settings below mostly use default values. No extensive hyperparameter tuning was performed, as the aim was to compare the general behaviour of the recommenders rather than to fully optimize them.

In [14]:
hybrid_rec = HybridItemKNNRecommender(
    collaborative_weight=0.75,
    content_weight=0.25,
    neutral_rating=6.0,
    explicit_weight=1,
    implicit_weight=0.25,
    component_norm="maxabs",
    diversity_lambda=0.9,
    rerank_pool=150,
)
hybrid_rec.fit(books, ratings)

In [9]:
user_knn_rec = UserKNNRecommender(
    explicit_min=1.0,
    min_user_ratings=10,
    min_item_ratings=10,
    k_neighbors=50,
    like_rating=9.0,
    similarity_mode="full",
    neighbors_only=False
)
user_knn_rec.fit(books, ratings)

In [10]:
mf_rec = BiasedMFRecommender(
    explicit_min=1.0,
    min_user_ratings=10,
    min_item_ratings=10,
    k_factors=30,
    lr=0.01,
    reg=0.05,
    n_epochs=40,
    like_rating=9.0,
)
mf_rec.fit(books, ratings)

## Public API

All recommenders below expose the same two public methods:

- `recommend_by_ids(seed_book_ids, ...)`
- `recommend_by_title(seed_titles, ...)`

For the explicit-feedback models, you can also pass `seed_ratings=[...]` to describe how strongly the query user likes each seed book.

The hybrid model accepts `seed_ratings` as well, but interprets them as seed weights rather than as direct rating targets.

In [11]:
# Example with one seed ID
display(hybrid_rec.recommend_by_ids("202812", n=10))
print()

# Example with multiple seed IDs
display(user_knn_rec.recommend_by_ids(["82690", "189376", "202812"], seed_ratings=[9, 5, 7], n=10))
print()

# Example with titles
display(mf_rec.recommend_by_title(
    ["Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))", "The Da Vinci Code"],
    seed_ratings=[9, 8],
    n=10,
))


,book_id,title,author,publisher,year,isbn,image_url,explicit_ratings,avg_explicit_rating,score
0,156614,Roverandom,J.R.R. Tolkien,Houghton Mifflin,1999,0395957990,http://images.amazon.com/images/P/0395957990.0...,4,8.250000,0.247
1,202813,"The Lord of the rings,",J. R. R Tolkien,Allen and Unwin,1968,0048230871,http://images.amazon.com/images/P/0048230871.0...,6,9.833333,0.227
2,202805,The Lord of the Rings (One Volume Edition),J. R. R. Tolkien,Houghton Mifflin Company,1999,0395974682,http://images.amazon.com/images/P/0395974682.0...,4,7.500000,0.205
3,114354,Lord of the Rings Trilogy,J.R.R. Tolkien,Houghton Mifflin,2001,0739408259,http://images.amazon.com/images/P/0739408259.0...,2,10.000000,0.203
4,202801,The Lord of the Rings (BBC Dramatization),J.R.R. TOLKIEN,Random House Audio,1999,0553456539,http://images.amazon.com/images/P/0553456539.0...,2,9.500000,0.189
5,202802,The Lord of the Rings (Leatherette Collector's...,J. R. R. Tolkien,Houghton Mifflin Company,1974,0395193958,http://images.amazon.com/images/P/0395193958.0...,6,10.000000,0.187
6,202815,The Lord of the Rings: The QPB Companion to th...,J.R.R. Tolkien,Quality Paper Back Book Club,2001,0965307883,http://images.amazon.com/images/P/0965307883.0...,3,8.000000,0.184
7,202803,The Lord of the Rings (Movie Art Cover),J. R. R. Tolkien,Houghton Mifflin Company,2002,0618260242,http://images.amazon.com/images/P/0618260242.0...,2,9.000000,0.182
8,202804,The Lord of the Rings (Movie Art Cover),J.R.R. Tolkien,Houghton Mifflin Company,2001,0618129022,http://images.amazon.com/images/P/0618129022.0...,38,9.000000,0.182
9,219189,"The Two Towers (The Lord of the Rings, Part 2)",J. R. R. Tolkien,Houghton Mifflin Company,1999,0618002235,http://images.amazon.com/images/P/0618002235.0...,25,9.720000,0.179


,book_id,title,author,publisher,year,isbn,image_url,explicit_ratings,avg_explicit_rating,score
0,213516,The Screwtape Letters,C. S. Lewis,Harper SanFrancisco,2001,0060652934,http://images.amazon.com/images/P/0060652934.0...,18,7.777778,10.000
1,87025,Holes (Yearling Newbery),LOUIS SACHAR,Yearling,2000,0440414806,http://images.amazon.com/images/P/0440414806.0...,76,8.657895,10.000
2,91471,I Know This Much Is True,Wally Lamb,Regan Books,1999,0060987561,http://images.amazon.com/images/P/0060987561.0...,94,8.265957,9.920
3,137972,One Day in the Life of Ivan Denisovich (Signet...,Alexander Solzhenitsyn,Signet Book,1998,0451527097,http://images.amazon.com/images/P/0451527097.0...,10,8.000000,9.920
4,138153,One Hundred Years of Solitude (Oprah's Book Club),Gabriel Garcia Marquez,Perennial,2004,0060740450,http://images.amazon.com/images/P/0060740450.0...,31,6.516129,9.920
5,136964,"Old Possum's Book of Practical Cats, Illustrat...",T.S. Eliot,Harvest Books,1982,015668568X,http://images.amazon.com/images/P/015668568X.0...,15,8.800000,9.769
6,90525,Human Croquet,Kate Atkinson,Picador USA,1998,0312186886,http://images.amazon.com/images/P/0312186886.0...,8,8.125000,9.769
7,198352,The Hours : A Novel,Michael Cunningham,Picador,2000,0312243022,http://images.amazon.com/images/P/0312243022.0...,73,7.520548,9.769
8,22027,Behind the Scenes at the Museum : A Novel,Kate Atkinson,Picador,1999,0312150601,http://images.amazon.com/images/P/0312150601.0...,23,7.652174,9.769
9,114328,Lord of the Flies,William Golding,Riverhead Books,1997,1573226122,http://images.amazon.com/images/P/1573226122.0...,13,8.384615,9.696


,book_id,title,author,publisher,year,isbn,image_url,explicit_ratings,avg_explicit_rating,score
0,53510,Dilbert: A Book of Postcards,Scott Adams,Andrews McMeel Pub,1996,0836213319,http://images.amazon.com/images/P/0836213319.0...,13,9.923077,9.640
1,156597,Route 66 Postcards: Greetings from the Mother ...,Michael Wallis,St. Martin's Press,1993,0312099045,http://images.amazon.com/images/P/0312099045.0...,11,9.727273,9.530
2,82690,Harry Potter and the Chamber of Secrets Postca...,J. K. Rowling,Scholastic,2002,0439425220,http://images.amazon.com/images/P/0439425220.0...,23,9.869565,9.401
3,72246,Fox in Socks (I Can Read It All by Myself Begi...,Dr. Seuss,Random House Children's Books,1965,0394800389,http://images.amazon.com/images/P/0394800389.0...,14,9.785714,9.380
4,183833,The Blue Day Book: A Lesson in Cheering Yourse...,Bradley Trevor Greive,Random House Australia,<NA>,0091842050,http://images.amazon.com/images/P/0091842050.0...,11,9.181818,9.365
5,130240,"My Sister's Keeper : A Novel (Picoult, Jodi)",Jodi Picoult,Atria,2004,0743454529,http://images.amazon.com/images/P/0743454529.0...,22,9.545455,9.339
6,146765,Postmarked Yesteryear: 30 Rare Holiday Postcards,Pamela E. Apkarian-Russell,Collectors Press,2001,1888054557,http://images.amazon.com/images/P/1888054557.0...,11,10.000000,9.300
7,121019,Maus a Survivors Tale: My Father Bleeds History,Art Spiegelman,Pantheon Books,1986,0394747232,http://images.amazon.com/images/P/0394747232.0...,16,9.312500,9.275
8,36258,Chronicle of a Death Foretold,GABRIEL GARCIA MARQUEZ,Ballantine Books,1984,0345310020,http://images.amazon.com/images/P/0345310020.0...,16,8.187500,9.258
9,152706,Redeeming Love,Francine Rivers,Multnomah,2001,1576738167,http://images.amazon.com/images/P/1576738167.0...,11,9.272727,9.248


## Example recommendations

The following cells show example queries using both book IDs and book titles.


In [15]:
seed_ids = ["82717", "189376", "202812"]
# 82717: Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))
# 189376: The Da Vinci Code
# 202812: The Lord of the Rings

print("HYBRID")
display(hybrid_rec.recommend_by_ids(seed_ids, n=10, rerank_diversity=True, mmr_lambda=0.7, candidate_pool=100))
print()

print("USER KNN")
display(user_knn_rec.recommend_by_ids(seed_ids, seed_ratings=[9, 5, 7], n=10))
print()

print("USER MF")
display(mf_rec.recommend_by_ids(seed_ids, seed_ratings=[9, 5, 7], n=10))



HYBRID


,book_id,title,author,publisher,year,isbn,image_url,explicit_ratings,avg_explicit_rating,score
0,82686,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,Scholastic,2000,0439064872,http://images.amazon.com/images/P/0439064872.0...,189,8.783069,10.001
1,82694,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,Scholastic,2000,0439139597,http://images.amazon.com/images/P/0439139597.0...,137,9.262774,9.001
2,156614,Roverandom,J.R.R. Tolkien,Houghton Mifflin,1999,0395957990,http://images.amazon.com/images/P/0395957990.0...,4,8.250000,8.001
3,82706,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,Scholastic,1999,0439136350,http://images.amazon.com/images/P/0439136350.0...,141,9.035461,7.001
4,14737,Angels &amp; Demons,Dan Brown,Pocket Star,2001,0671027360,http://images.amazon.com/images/P/0671027360.0...,269,8.100372,6.001
5,82699,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,Scholastic,2003,043935806X,http://images.amazon.com/images/P/043935806X.0...,206,9.033981,5.001
6,60234,El Codigo Da Vinci / The Da Vinci Code,Dan Brown,Ediciones Urano,2003,8495618605,http://images.amazon.com/images/P/8495618605.0...,20,7.900000,4.000
7,82713,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,Scholastic,1998,0590353403,http://images.amazon.com/images/P/0590353403.0...,119,8.983193,3.001
8,83765,Heart of Darkness and Other Tales (Oxford Worl...,Joseph Conrad,Oxford University Press,1998,0192833731,http://images.amazon.com/images/P/0192833731.0...,1,10.000000,2.000
9,198610,The Human Nature of Birds: A Scientific Discov...,"Theodore Xenophon, Ph.D. Barber",Penguin USA,1994,0140234942,http://images.amazon.com/images/P/0140234942.0...,1,10.000000,1.000



USER KNN


,book_id,title,author,publisher,year,isbn,image_url,explicit_ratings,avg_explicit_rating,score
0,180728,The Analyst,John Katzenbach,Ballantine Books,2003,0345426274,http://images.amazon.com/images/P/0345426274.0...,9,7.666667,10.000
1,146529,Portrait in Sepia : A Novel,Isabel Allende,Perennial,2002,0060936363,http://images.amazon.com/images/P/0060936363.0...,26,8.230769,10.000
2,41775,Cowboys Are My Weakness,Pam Houston,Washington Square Press,1993,0671793888,http://images.amazon.com/images/P/0671793888.0...,14,7.428571,10.000
3,213516,The Screwtape Letters,C. S. Lewis,Harper SanFrancisco,2001,0060652934,http://images.amazon.com/images/P/0060652934.0...,18,7.777778,10.000
4,72428,Frankenstein (Changing Our World),Mary Wollstonecraft Shelley,Bantam,1984,0553212478,http://images.amazon.com/images/P/0553212478.0...,28,7.785714,10.000
5,94506,In the Heart of the Sea: The Tragedy of the Wh...,Nathaniel Philbrick,Penguin Putnam,2000,0670891576,http://images.amazon.com/images/P/0670891576.0...,19,7.894737,10.000
6,137972,One Day in the Life of Ivan Denisovich (Signet...,Alexander Solzhenitsyn,Signet Book,1998,0451527097,http://images.amazon.com/images/P/0451527097.0...,10,8.000000,9.920
7,138153,One Hundred Years of Solitude (Oprah's Book Club),Gabriel Garcia Marquez,Perennial,2004,0060740450,http://images.amazon.com/images/P/0060740450.0...,31,6.516129,9.920
8,186983,The Clan of the Cave Bear (Earth's Children (P...,Jean M. Auel,Bantam Books,1984,0553250426,http://images.amazon.com/images/P/0553250426.0...,89,8.213483,9.846
9,22027,Behind the Scenes at the Museum : A Novel,Kate Atkinson,Picador,1999,0312150601,http://images.amazon.com/images/P/0312150601.0...,23,7.652174,9.769



USER MF


,book_id,title,author,publisher,year,isbn,image_url,explicit_ratings,avg_explicit_rating,score
0,72246,Fox in Socks (I Can Read It All by Myself Begi...,Dr. Seuss,Random House Children's Books,1965,0394800389,http://images.amazon.com/images/P/0394800389.0...,14,9.785714,8.313
1,245986,Wuthering Heights,EMILY BRONTE,Bantam,1983,0553212583,http://images.amazon.com/images/P/0553212583.0...,28,8.071429,8.246
2,198022,The Hobbit: or There and Back Again,J.R.R. Tolkien,Houghton Mifflin Company,1999,0618002219,http://images.amazon.com/images/P/0618002219.0...,39,9.000000,8.241
3,53510,Dilbert: A Book of Postcards,Scott Adams,Andrews McMeel Pub,1996,0836213319,http://images.amazon.com/images/P/0836213319.0...,13,9.923077,8.224
4,82690,Harry Potter and the Chamber of Secrets Postca...,J. K. Rowling,Scholastic,2002,0439425220,http://images.amazon.com/images/P/0439425220.0...,23,9.869565,8.219
5,156597,Route 66 Postcards: Greetings from the Mother ...,Michael Wallis,St. Martin's Press,1993,0312099045,http://images.amazon.com/images/P/0312099045.0...,11,9.727273,8.206
6,213860,The Secret Garden,Frances Hodgson Burnett,HarperTrophy,1998,006440188X,http://images.amazon.com/images/P/006440188X.0...,20,9.200000,8.199
7,185456,The Calvin and Hobbes Tenth Anniversary Book,Bill Watterson,Andrews McMeel Publishing,1995,0836204387,http://images.amazon.com/images/P/0836204387.0...,23,9.260870,8.112
8,82694,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,Scholastic,2000,0439139597,http://images.amazon.com/images/P/0439139597.0...,137,9.262774,8.105
9,121019,Maus a Survivors Tale: My Father Bleeds History,Art Spiegelman,Pantheon Books,1986,0394747232,http://images.amazon.com/images/P/0394747232.0...,16,9.312500,8.083


In [16]:
seed_titles = [
    "Dune",
    "The Hobbit: or There and Back Again",
    "The Lord of the Rings",
]

print("HYBRID")
display(hybrid_rec.recommend_by_title(seed_titles, n=10, rerank_diversity=True, mmr_lambda=0.8, candidate_pool=100))

print("USER KNN")
display(user_knn_rec.recommend_by_title(seed_titles, seed_ratings=[9, 10, 8], n=10))

print("USER MF")
display(mf_rec.recommend_by_title(seed_titles, seed_ratings=[9, 10, 8], n=10))


HYBRID


,book_id,title,author,publisher,year,isbn,image_url,explicit_ratings,avg_explicit_rating,score
0,212008,"The Return of the King (The Lord of The Rings,...",J. R. R. Tolkien,Houghton Mifflin Company,1999,0618002243,http://images.amazon.com/images/P/0618002243.0...,16,9.625000,10.001
1,35324,Children of Dune,Frank Herbert,Berkley Publishing Group,1977,0425033104,http://images.amazon.com/images/P/0425033104.0...,2,8.500000,9.001
2,231086,Ultimate Japanese: Advanced (Living Language U...,SUGURU AKUTSU,Living Language,1998,0517885042,http://images.amazon.com/images/P/0517885042.0...,1,10.000000,8.001
3,43876,Cyoa Lost Jewels of Nabooti,R.a Montgomery,Bantam Doubleday Dell,<NA>,0553143581,http://images.amazon.com/images/P/0553143581.0...,3,6.666667,7.001
4,203462,The Magic School Bus: Inside the Earth,Joanna Cole,Scholastic,1987,0590407597,http://images.amazon.com/images/P/0590407597.0...,1,10.000000,6.001
5,217035,The Super-Duper Sleepover Party (Full House Mi...,R.L. Stine,Simon Spotlight,1995,0671519069,http://images.amazon.com/images/P/0671519069.0...,1,10.000000,5.001
6,91539,I Love to Read! (Four Original Stories for Ind...,Rachel Vail,Scholastic,2002,0439448328,http://images.amazon.com/images/P/0439448328.0...,2,7.500000,4.001
7,16409,ARE WE HAVING FUN YET ACTIVITIES INSPIRED BY N...,Carmen Morais,Aladdin,1998,0671016822,http://images.amazon.com/images/P/0671016822.0...,1,10.000000,3.001
8,130306,MY TEACHER IS AN ALIEN : MY TEACHER IS AN ALIEN,Bruce Coville,Aladdin,1990,0671737295,http://images.amazon.com/images/P/0671737295.0...,2,8.500000,2.001
9,4469,A Hope in the Unseen : An American Odyssey fro...,RON SUSKIND,Broadway,1999,0767901266,http://images.amazon.com/images/P/0767901266.0...,5,8.800000,1.001


USER KNN


,book_id,title,author,publisher,year,isbn,image_url,explicit_ratings,avg_explicit_rating,score
0,22223,Bell Jar,Sylvia Plath,Faber Faber Inc,<NA>,0571081789,http://images.amazon.com/images/P/0571081789.0...,29,8.068966,10.0
1,190139,The Defiant Hero,SUZANNE BROCKMANN,Ivy Books,2001,0804119538,http://images.amazon.com/images/P/0804119538.0...,9,8.222222,10.0
2,23425,Between Friends,Debbie Macomber,Mira Books,2003,155166674X,http://images.amazon.com/images/P/155166674X.0...,29,7.620690,10.0
3,110192,Letters from a Nut,Ted L. Nancy,William Morrow,1999,0380973545,http://images.amazon.com/images/P/0380973545.0...,29,7.517241,10.0
4,227761,Touching Evil,Kay Hooper,Bantam Books,2001,0553583441,http://images.amazon.com/images/P/0553583441.0...,29,7.482759,10.0
5,103200,Kitchen Table Wisdom: Stories That Heal,Rachel Naomi Remen,Riverhead Books,1997,1573226106,http://images.amazon.com/images/P/1573226106.0...,9,8.555556,10.0
6,78031,Going After Cacciato,Tim O'Brien,Broadway Books,1999,0767904427,http://images.amazon.com/images/P/0767904427.0...,9,8.666667,10.0
7,167033,Small Town Girl,Lavyrle Spencer,Jove Books,1998,051512219X,http://images.amazon.com/images/P/051512219X.0...,29,7.275862,10.0
8,65013,Extension Du Domain De La Lutte,Michel Houellebecq,European Book Co,2002,2290045764,http://images.amazon.com/images/P/2290045764.0...,12,7.916667,10.0
9,186098,The Cat Who Went Up the Creek,Lilian Jackson Braun,Jove Books,2003,0515134384,http://images.amazon.com/images/P/0515134384.0...,29,6.965517,10.0


USER MF


,book_id,title,author,publisher,year,isbn,image_url,explicit_ratings,avg_explicit_rating,score
0,130240,"My Sister's Keeper : A Novel (Picoult, Jodi)",Jodi Picoult,Atria,2004,0743454529,http://images.amazon.com/images/P/0743454529.0...,22,9.545455,9.838
1,53510,Dilbert: A Book of Postcards,Scott Adams,Andrews McMeel Pub,1996,0836213319,http://images.amazon.com/images/P/0836213319.0...,13,9.923077,9.810
2,82690,Harry Potter and the Chamber of Secrets Postca...,J. K. Rowling,Scholastic,2002,0439425220,http://images.amazon.com/images/P/0439425220.0...,23,9.869565,9.779
3,156597,Route 66 Postcards: Greetings from the Mother ...,Michael Wallis,St. Martin's Press,1993,0312099045,http://images.amazon.com/images/P/0312099045.0...,11,9.727273,9.775
4,121014,Maus 1. Mein Vater kotzt Geschichte aus. Die G...,Art Spiegelman,Rowohlt Tb.,1999,3499224615,http://images.amazon.com/images/P/3499224615.0...,10,9.700000,9.688
5,72246,Fox in Socks (I Can Read It All by Myself Begi...,Dr. Seuss,Random House Children's Books,1965,0394800389,http://images.amazon.com/images/P/0394800389.0...,14,9.785714,9.677
6,185456,The Calvin and Hobbes Tenth Anniversary Book,Bill Watterson,Andrews McMeel Publishing,1995,0836204387,http://images.amazon.com/images/P/0836204387.0...,23,9.260870,9.655
7,218259,The Time Traveler's Wife,Audrey Niffenegger,MacAdam/Cage Publishing,2003,193156146X,http://images.amazon.com/images/P/193156146X.0...,25,9.120000,9.650
8,82717,Harry Potter and the Sorcerer's Stone (Harry P...,J. K. Rowling,Arthur A. Levine Books,1999,059035342X,http://images.amazon.com/images/P/059035342X.0...,313,8.939297,9.591
9,121019,Maus a Survivors Tale: My Father Bleeds History,Art Spiegelman,Pantheon Books,1986,0394747232,http://images.amazon.com/images/P/0394747232.0...,16,9.312500,9.548
